In [67]:
import numpy as np
from scipy.io import loadmat
import scipy.io as sio
import os
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu, levene, shapiro, ttest_ind
from statsmodels.tsa.stattools import adfuller
from scipy.signal import find_peaks
import pandas as pd

***PUNTO 2.***

Se cuenta con señales de EEG de dos grupos de personas, un grupo control y un grupo de pacientes con enfermedad de Parkinson. Se tiene evidencia que la energía de las señales de EEG puede conducir a diferenciar entre señales de pacientes con enfermedad de Parkinson y sanos. Se pide calcular la Energía
promedio por grupo poblacional de cada canal.

***Teoría***

Un electroencefalograma es una prueba que mide la actividad eléctrica del cerebro. Esta prueba requiere la colocación en el cuero cabelludo de electrodos, que son pequeños discos metálicos. Las neuronas cerebrales se comunican mediante impulsos eléctricos, y esta actividad se manifiesta como líneas onduladas en un registro electroencefalográfico (Mayo Clinic, 2023).

La enfermedad de Parkinson es un trastorno neurodegenerativo que genera alteraciones en la actividad eléctrica cerebral detectables mediante EEG. La energía de la señal, definida como la suma de los cuadrados de sus muestras, es una característica que cuantifica la actividad total de cada canal. 
Aljalal et al. (2022) propusieron métodos basados en patrones espaciales comunes para detectar Parkinson en señales EEG en reposo, extrayendo características como potencia de banda, energía y distintos tipos de entropía para distinguir pacientes con Parkinson de controles sanos, alcanzando 
precisiones de clasificación superiores al 99%. Sus resultados mostraron además que las bandas alfa (8-13 Hz, reposo y ojos cerrados) y beta (13-30 Hz, estado de alerta/conectración/actividad motora) son las que mayor información discriminativa aportan entre ambos grupos poblacionales.

El EEG se adquiere digitalmente, lo que significa que la señal continua del cerebro es muestreada 
a intervalos regulares de tiempo, produciendo una secuencia discreta de valores. Para señales 
discretas, la energía se define como la suma de los cuadrados de sus muestras (Alexander & Williams, 2017):

E = Σ |x(n)|²

En este trabajo, la energía se calcula para cada época y luego se promedia entre todas las épocas 
del sujeto, obteniendo un único valor representativo por canal. Esto permite comparar sujetos 
independientemente del número de épocas registradas.



***Desarrollo***

Primero se busca reconocer la estructura de los archivos .mat, identificando que el diccionario con llave "data" contiene los datos de interés. Luego, identificamos las dimensiones del arreglo de data, siendo de 8 canales, 2000 muestras y las épocas varían entre sujetos. Esta variación muestra que a cada sujeto se le grabó por un tiempo distinto, quien tuvo más tiempo de grabación tiene más épocas. 

Por otra parte, la energía se calcula por época y luego se promedia, así que cada sujeto está representado por un solo valor por canal, indpendientemete de cuantas épocas tenga.|

In [68]:
mat = loadmat(r'C:\Users\valen\Downloads\Practuca3Punto2\control\C001R_EP_reposo.mat')

#.items() para conocer nuevamente las llaves y valores del diccionario. En este caso nos intereresa "data"
for clave, valor in mat.items(): 
    print(clave, ':', type(valor))

__header__ : <class 'bytes'>
__version__ : <class 'str'>
__globals__ : <class 'list'>
data : <class 'numpy.ndarray'>


In [69]:
archivos_parkinson = os.listdir(r'C:\Users\valen\Downloads\Practuca3Punto2\parkinson')

for archivo in archivos_parkinson:
    mat = loadmat(r'C:\Users\valen\Downloads\Practuca3Punto2\parkinson' + '\\' + archivo)
    print(archivo, ':', mat['data'].shape)

P001_EP_reposo.mat : (8, 2000, 143)
P004_EP_reposo.mat : (8, 2000, 138)
P005_EP_reposo.mat : (8, 2000, 174)
P007_EP_reposo.mat : (8, 2000, 153)
P012_EP_reposo.mat : (8, 2000, 140)
P013_EP_reposo.mat : (8, 2000, 143)
P015_EP_reposo.mat : (8, 2000, 147)
P016_EP_reposo.mat : (8, 2000, 150)
P017_EP_reposo.mat : (8, 2000, 176)
P018_EP_reposo.mat : (8, 2000, 149)
P020_EP_reposo.mat : (8, 2000, 140)
P025_EP_reposo.mat : (8, 2000, 154)
P026_EP_reposo.mat : (8, 2000, 142)
P028_EP_reposo.mat : (8, 2000, 174)
P030_EP_reposo.mat : (8, 2000, 171)
P032_EP_reposo.mat : (8, 2000, 164)
P033_EP_reposo.mat : (8, 2000, 154)
P034_EP_reposo.mat : (8, 2000, 159)
P040_EP_reposo.mat : (8, 2000, 198)
P041_EP_reposo.mat : (8, 2000, 157)
P046_EP_reposo.mat : (8, 2000, 167)
P048_EP_reposo.mat : (8, 2000, 162)
P049_EP_reposo.mat : (8, 2000, 170)


In [70]:
# os.listdir() devuelve una lista con todos los nombres de archivo dentro de la carpeta
archivos_control = os.listdir(r'C:\Users\valen\Downloads\Practuca3Punto2\control')

# Recorre cada archivo de la carpeta control
for archivo in archivos_control:
    # Carga el archivo .mat y lo convierte en diccionario
    mat = loadmat(r'C:\Users\valen\Downloads\Practuca3Punto2\control' + '\\' + archivo)
    # Imprime el nombre del archivo y el shape de su campo data
    print(archivo, '→', mat['data'].shape)

C001R_EP_reposo.mat → (8, 2000, 180)
C002_EP_reposo.mat → (8, 2000, 176)
C004_EP_reposo.mat → (8, 2000, 146)
C005_EP_reposo_Repetido.mat → (8, 2000, 149)
C006_EP_reposo.mat → (8, 2000, 168)
C007_EP_reposo.mat → (8, 2000, 173)
C010_EP_reposo.mat → (8, 2000, 136)
C011_EP_reposo.mat → (8, 2000, 173)
C012_EP_reposo.mat → (8, 2000, 172)
C013_EP_reposo.mat → (8, 2000, 175)
C015_EP_reposo.mat → (8, 2000, 140)
C018_EP_reposo.mat → (8, 2000, 157)
C019_EP_reposo.mat → (8, 2000, 179)
C020_EP_reposo.mat → (8, 2000, 166)
C021_EP_reposo.mat → (8, 2000, 180)
C023_EP_reposo.mat → (8, 2000, 206)
C024_EP_reposo.mat → (8, 2000, 186)
C025_EP_reposo.mat → (8, 2000, 154)
C026_EP_reposo.mat → (8, 2000, 154)
C027_EP_reposo.mat → (8, 2000, 142)
C028_EP_reposo.mat → (8, 2000, 169)
C029_EP_reposo.mat → (8, 2000, 160)
C030_EP_reposo.mat → (8, 2000, 144)
C031_EP_reposo.mat → (8, 2000, 178)
C032_EP_reposo.mat → (8, 2000, 163)
C033R_EP_reposo.mat → (8, 2000, 189)
C034_EP_reposo.mat → (8, 2000, 173)
C036_EP_reposo.ma

***1. Implemente una función que reciba una señal de múltiples canales y épocas y calcule la Energía de promedio de cada canal.***

In [71]:
def energia(data):
    energia_por_epoca = np.sum(data ** 2, axis=1) #suma todos los valores al cuadrado a lo largo del eje de muestras (axis=1), sale una arreglo de (canales, épocas)                          
    energia_promedio = np.mean(energia_por_epoca, axis=1) #se obtiene un arreglo 1D array de (canales,), es decir, un arreglo de 8 valores, uno por canal
    return energia_promedio  


***2. Calcule la energía de cada canal promediada por épocas para cada sujeto, esto para ambos grupos poblacionales. Guarde esta información en un DataFrame de columnas ‘canal’ y filas ‘#sujeto’ con los valores de energía calculados, un DataFrame para cada grupo poblacional. 

Primero se obtiene una lista de 8 valores por canal para cada sujeto. Estas listas se convierten en matriz que permite crear el DataFrame.***

In [72]:
#os.listdir() devuelve la lista de nombres de archivos dentro de la carpeta
archivos_control   = os.listdir(r'C:\Users\valen\Downloads\Practuca3Punto2\control')
archivos_parkinson = os.listdir(r'C:\Users\valen\Downloads\Practuca3Punto2\parkinson')
resultados_control   = [] #listas que almacenan los resultados de energía de cada sujeto que se usarán en el DataFrame
resultados_parkinson = []

#Se realizan ciclos para extraer (canales, muestras, épocas) de la data de cada sujeto
for archivo in archivos_control:
    mat = loadmat(r'C:\Users\valen\Downloads\Practuca3Punto2\control' + '\\' + archivo)
    data = mat['data']
    prom_energia = energia(data) #calcula la energía promedio por canal para este sujeto
    print(prom_energia)
    resultados_control.append(prom_energia) #agregar los resultados a las listas

#Se extrean los datos de igual forma para el grupo de Parkinson
for archivo in archivos_parkinson:
    mat = loadmat(r'C:\Users\valen\Downloads\Practuca3Punto2\parkinson' + '\\' + archivo)
    data = mat['data']
    calc_energia = energia(data)
    print(calc_energia)
    resultados_parkinson.append(calc_energia)

[21465.65035816 20985.90791202 22760.14958825 18505.64028363
 29730.16302581 25244.15807256 22781.32758731 24658.59951166]
[15966.40286835 17617.81024822 20804.93712863 19654.40001721
 16678.98206306 93894.04900934 66862.49627529 75685.12587166]
[14148.67332164 18283.99966574 28749.93214817 14270.72691057
 28787.4459783  14661.41773957 15940.15409495 19499.89865573]
[ 35311.30169608  34916.6860104   38800.42902907  35427.03112743
  35905.47286877 106598.12815176 106885.57596622 112520.75063632]
[18510.82997904 19738.4893753  20911.79274757 21828.25439857
 23351.99264925 53086.05976598 37495.9724752  43067.09550443]
[13180.10931685 13925.21781249 16218.99422301 12324.88365918
 14060.30065885 25767.02486391 21935.75962238 22827.7812929 ]
[11197.55457374 10948.36880483 12737.0046647  10745.16192106
 10329.64241838 21461.60583118 15493.21277613 27414.37524728]
[28551.1240649  26204.83925352 17383.99895593 17244.60593265
 26206.37242245 83370.61840752 51121.10572221 67852.34825552]
[ 9133.0

Construcción de DataFrame para cada grupo (control y parkinson)

In [73]:
matriz_control   = np.array(resultados_control) #np.array() convierte la lista de resultados en una matriz de dimensiones (sujetos, canales)
matriz_parkinson = np.array(resultados_parkinson)

columnas = [] #Crea la lista de nombres de columnas: ['canal_0', 'canal_1', ..., 'canal_7']
for i in range(8):
    columnas.append(f'canal_{i}')

#pd.DataFrame() organiza la matriz en una tabla. 
df_control   = pd.DataFrame(matriz_control,   columns=columnas)
df_parkinson = pd.DataFrame(matriz_parkinson, columns=columnas)

indices_control = [] #lista para agregar de nombres de filas: ['sujeto_1', 'sujeto_2', ...]
for i in range(len(df_control)): 
    indices_control.append(f'sujeto_{i+1}')

indices_parkinson = []
for i in range(len(df_parkinson)):
    indices_parkinson.append(f'sujeto_{i+1}')

df_control.index   = indices_control #Líneas para nombrar el número de sujeto
df_parkinson.index = indices_parkinson  

print('DataFrame control:')
print(df_control)
print('\nDataFrame parkinson:')
print(df_parkinson)

# .mean() calcula el promedio de energía de cada canal a lo largo de todos los sujetos
energia_media_control   = df_control.mean()
energia_media_parkinson = df_parkinson.mean()

# .idxmax() devuelve el nombre del canal con mayor valor promedio de energía
# .idxmin() devuelve el nombre del canal con menor valor promedio de energía
print('Control')
print(f'Canal con más energía:  {energia_media_control.idxmax()} ({energia_media_control.max():.2f})')
print(f'Canal con menos energía: {energia_media_control.idxmin()} ({energia_media_control.min():.2f})')

print('\nParkinson')
print(f'Canal con más energía:  {energia_media_parkinson.idxmax()} ({energia_media_parkinson.max():.2f})')
print(f'Canal con menos energía: {energia_media_parkinson.idxmin()} ({energia_media_parkinson.min():.2f})')



DataFrame control:
                canal_0       canal_1       canal_2       canal_3  \
sujeto_1   21465.650358  20985.907912  22760.149588  18505.640284   
sujeto_2   15966.402868  17617.810248  20804.937129  19654.400017   
sujeto_3   14148.673322  18283.999666  28749.932148  14270.726911   
sujeto_4   35311.301696  34916.686010  38800.429029  35427.031127   
sujeto_5   18510.829979  19738.489375  20911.792748  21828.254399   
sujeto_6   13180.109317  13925.217812  16218.994223  12324.883659   
sujeto_7   11197.554574  10948.368805  12737.004665  10745.161921   
sujeto_8   28551.124065  26204.839254  17383.998956  17244.605933   
sujeto_9    9133.036290   9214.155028  11626.411811  10809.621612   
sujeto_10  47166.556798  55107.798641  52286.884667  34682.656928   
sujeto_11  17567.465030  21738.511853  29429.308030  28530.200793   
sujeto_12  31250.507507  24222.776323  28298.111428  25378.777621   
sujeto_13  34036.502777  35276.242239  37728.034174  30403.012785   
sujeto_14   491

**Nota: los resultados son largos, por lo que se ven completos con la opción de abrir en un editor de texto**

***3. Determine si existe diferencia estadística entre canales de cada grupo de sujetos a través de una prueba t. Compruebe los supuestos necesarios para realizar una prueba t, esto es: Normalidad de la variable, independencia (se asume que los grupos son independientes), y homocedasticidad (use una prueba de
Levene), finalmente realice la prueba t para determinar si existen diferencias entre los canales entre grupos de sujetos. De no cumplirse los requisitos, realice entonces un análisis no paramétrico (pruebaU de Mann-Whitney). Este numeral tiene como objetivo identificar los canales que entregan información diferencial entre pacientes Sanos y con enfermedad de Parkinson.***

In [74]:
#se recorre cada uno de los 8 canales para comparar entre grupos y se extraen sus valores para el canal i
for i in range(8):
    print(f'Canal {i}:')
    grupo_control   = df_control[f'canal_{i}'].values #array con un valor por sujeto de control
    grupo_parkinson = df_parkinson[f'canal_{i}'].values  #array con un valor por sujeto de control parkinson
    
    #supuesto 1: normalidad
    _, p_normal_control   = shapiro(grupo_control) #por medio de Shapiro rechazamos o no la hipótesis de que los datos son normales
    _, p_normal_parkinson = shapiro(grupo_parkinson)
    print(f'Shapiro control   p={p_normal_control:.4f}  : {"Normal" if p_normal_control > 0.05 else "No normal"}')
    print(f'Shapiro parkinson p={p_normal_parkinson:.4f}  : {"Normal" if p_normal_parkinson > 0.05 else "No normal"}')
    
    #supuesto 2: Homocedasticidad, probar si ambos grupos tienen varianzas iguales
    # Hipótesis nula: las varianzas son iguales. Si p > 0.05, no se rechaza H0 así que hay homocedasticidad
    _, p_levene = levene(grupo_control, grupo_parkinson) # Levene entrega un valor estadístico y el valor p, por lo que con "__" no se almacena ese primer valor
    print(f'Levene            p={p_levene:.4f}  : {"Homocedástico" if p_levene > 0.05 else "Heterocedástico"}')

    #luego de correr el código observamos que las muestras no son normales, por lo que se selcciona la prueba Mann-Whitney.
    # Hipótesis nula: las distribuciones de ambos grupos son iguales
    # alternative='two-sided' evalúa diferencia en cualquier dirección, ya sea si el grupo control es mayor que parkinson o viceversa
    _, p_prueba = mannwhitneyu(grupo_control, grupo_parkinson, alternative='two-sided')
    print(f'Mann-Whitney U  p={p_prueba:.4f}  : {"Diferencia significativa ✓" if p_prueba < 0.05 else "Sin diferencia significativa"}')

Canal 0:
Shapiro control   p=0.0063  : No normal
Shapiro parkinson p=0.0144  : No normal
Levene            p=0.8848  : Homocedástico
Mann-Whitney U  p=0.4057  : Sin diferencia significativa
Canal 1:
Shapiro control   p=0.0040  : No normal
Shapiro parkinson p=0.0045  : No normal
Levene            p=0.9591  : Homocedástico
Mann-Whitney U  p=0.5705  : Sin diferencia significativa
Canal 2:
Shapiro control   p=0.0089  : No normal
Shapiro parkinson p=0.0009  : No normal
Levene            p=0.9776  : Homocedástico
Mann-Whitney U  p=0.4604  : Sin diferencia significativa
Canal 3:
Shapiro control   p=0.0002  : No normal
Shapiro parkinson p=0.0003  : No normal
Levene            p=0.7593  : Homocedástico
Mann-Whitney U  p=0.2345  : Sin diferencia significativa
Canal 4:
Shapiro control   p=0.0008  : No normal
Shapiro parkinson p=0.0053  : No normal
Levene            p=0.9046  : Homocedástico
Mann-Whitney U  p=0.5600  : Sin diferencia significativa
Canal 5:
Shapiro control   p=0.0000  : No normal
S

**Nota: los resultados son largos, por lo que se ven completos con la opción de abrir en un editor de texto**

***Análisis***

- Del numeral dos, se observa que la energía promedio cambia de canal a canal en ambos grupos, debido a la actividad cerebral. Al promediar la energía entre todos los sujetos para cada canal, se oberva que ambos grupos presentan el canal 7 con mayor energía y el 0 con menor:

    **Control**
    Canal con más energía:  canal_7 (92530.99)
    , canal con menos energía: canal_0 (22014.67)

    **Parkinson**
    Canal con más energía:  canal_7 (123540.59)
    , canal con menos energía: canal_0 (24349.67)

    Se observa además que el grupo Parkinson presenta valores de energía más altos en el canal 7, lo que podría sugerir mayor actividad eléctrica en esa derivación, aunque esta diferencia no resultó estadísticamente significativa en la prueba de Mann-Whitney del numeral 3.

- En el numeral 3, se puede decir que:

    **Normalidad (Shapiro-Wilk)**
    La prueba de Shapiro-Wilk rechazó la hipótesis de normalidad en todos los canales para ambos grupos (p < 0.05), indicando que los valores de energía no siguen una distribución normal. Por esta razón se descartó la prueba t de Student y se aplicó la prueba no paramétrica de Mann-Whitney.

    **Homocedasticidad (Levene)**
    La prueba de Levene no encontró diferencias significativas en las varianzas entre grupos en ningún canal (p > 0.05), confirmando homocedasticidad. Sin embargo, dado que el supuesto de normalidad no se cumplió, este resultado no fue suficiente para justificar el uso de la prueba t.

    **Prueba U de Mann-Whitney**
    La prueba U de Mann-Whitney no encontró diferencia estadísticamente significativa entre el grupo control y el grupo Parkinson en ninguno de los 8 canales (p > 0.05 en todos los casos). Una posible explicación es que la energía se calculó sobre la señal completa sin discriminar por bandas de frecuencia. Dado que las diferencias entre pacientes con Parkinson y sanos tienden a concentrarse en bandas específicas como alfa y beta (Aljalal et al., 2022), al integrar toda la energía de la señal estas diferencias quedan diluidas y no son detectables estadísticamente.


***Conclusiones***
- La energía promedio por canal, calculada como la suma de los cuadrados de las muestras promediada entre épocas, no permitió diferenciar estadísticamente entre sujetos sanos y pacientes con enfermedad de Parkinson en ninguno de los 8 canales analizados.

- Los datos de energía no siguieron una distribución normal en ningún canal ni en ningún grupo, lo que llevó al uso de la prueba no paramétrica de Mann-Whitney en lugar de la prueba t de Student.

- La energía total de la señal EEG puede no ser el descriptor más adecuado para diferenciar entre grupos. Un análisis por bandas de frecuencia específicas como alfa y beta podría ofrecer mayor poder discriminativo.

Referencias
1. Mayo Clinic. (2023). Electroencefalograma (EEG).
https://www.mayoclinic.org/es/tests-procedures/eeg/about/pac-20393875
2. Aljalal M, Aldosari SA, AlSharabi K, Abdurraqeeb AM, Alturki FA. Parkinson's Disease Detection from Resting-State EEG Signals Using Common Spatial Pattern, Entropy, and Machine Learning Techniques. Diagnostics (Basel). 2022 Apr 20;12(5):1033. doi: 10.3390/diagnostics12051033. PMID: 35626189; PMCID: PMC9139946.
3. Alexander, W., & Williams, C. (2017). Digital Signal Processing. Chapter 2. https://www.sciencedirect.com/science/chapter/monograph/abs/pii/B9780128045473000024 